<a href="https://colab.research.google.com/github/NAMUORI00/aerospace-rag/blob/main/notebooks/aerospace_rag_colab_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 항공우주 로컬 RAG Colab 노트북

이 노트북은 `smartfarm-workspace`에서 계승한 Dense/Sparse/Graph RAG 구조를 항공우주 문서용 Google Colab T4 데모로 정리한 학습형 실행 문서입니다. Colab을 실행할 때마다 GitHub에서 프로젝트를 새로 clone하고, `/content/aerospace-rag/data`에 둔 업무 파일을 읽어 로컬 인덱스를 만든 뒤, 근거가 표시되는 답변까지 확인합니다.

**목표**
- 기업 데모용: 항공우주 문서를 Colab T4에 올리고 RAG 답변, 상위 근거, 검색 진단을 한 화면에서 검토합니다.
- 학습용 구조 이해: chunk, vector store, BM25 문서, graph-lite entity/relation이 어떤 지식 데이터베이스 구조로 저장되는지 확인합니다.
- 검증용: 최소 세 개 이상의 업무형 질문에 대해 답변, source, retrieval diagnostics가 함께 남는지 확인합니다.

**실행 순서**
1. Colab 런타임을 GPU T4로 설정합니다.
2. 위에서 아래로 모든 셀을 순서대로 실행합니다.
3. 데이터 준비 섹션에서 지원 파일을 `/content/aerospace-rag/data` 아래에 둡니다.
4. 인덱스 생성 섹션에서 `qdrant`, `graph_index.json`, `bm25.json`, `chunks.jsonl` 산출물이 만들어졌는지 확인합니다.
5. 8A 섹션에서 도메인 데이터베이스 저장 구조와 지식 그래프 노드를 확인한 뒤 검색, 답변, 근거, 반복 질문 결과를 검토합니다.

**문제 대응**
- 패키지 import가 실패하면 의존성 설치 셀을 다시 실행합니다.
- Ollama가 응답하지 않으면 Ollama 런타임 셀을 다시 실행합니다.
- 데이터가 없다고 나오면 파일 위치가 `/content/aerospace-rag/data`인지 확인합니다.
- LLM 생성 없이 retrieval만 점검할 때는 4번 설정 셀에서 `ANSWER_PROVIDER = "extractive"`로 바꿉니다.

## 1. 실행 환경 확인

현재 런타임이 Colab인지 로컬인지, Python/OS/GPU 상태가 무엇인지 먼저 기록합니다. 이 정보는 같은 노트북을 다시 실행했을 때 환경 차이를 비교하는 기준입니다.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from IPython.display import HTML, Markdown, display

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def current_working_dir() -> Path:
    try:
        return Path.cwd()
    except FileNotFoundError:
        cwd_target = Path("/content") if Path("/content").exists() else Path.home()
        os.chdir(cwd_target)
        return Path.cwd()


RUN_STARTED_AT = time.strftime("%Y-%m-%d %H:%M:%S %Z", time.localtime())
print("Run started:", RUN_STARTED_AT)
print("Runtime:", "Colab" if IN_COLAB else "Local")
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
        stderr=subprocess.STDOUT,
    ).strip()
except Exception:
    gpu_info = "No GPU reported by nvidia-smi"
print("GPU:", gpu_info)
print("Initial cwd:", current_working_dir())

Run started: 2026-05-05 13:18:16 UTC
Runtime: Colab
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
GPU: Tesla T4, 15360 MiB
Initial cwd: /content


## 2. 프로젝트 소스 확보

Colab에서는 항상 `/content`로 이동한 뒤 기존 `/content/aerospace-rag`를 삭제하고 GitHub repo를 새로 clone합니다. clone 후 commit hash를 출력해 이번 실행이 어떤 코드 상태였는지 남깁니다.

In [2]:
GITHUB_REPO_URL = "https://github.com/NAMUORI00/aerospace-rag.git"
DEFAULT_COLAB_ROOT = Path("/content/aerospace-rag")

if IN_COLAB:
    os.chdir(DEFAULT_COLAB_ROOT.parent)
    print("Policy: Project source is cloned fresh for each Colab run.")
    print("Running git clone:", GITHUB_REPO_URL)
    if DEFAULT_COLAB_ROOT.exists():
        shutil.rmtree(DEFAULT_COLAB_ROOT)
    subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(DEFAULT_COLAB_ROOT)])
    os.chdir(DEFAULT_COLAB_ROOT)
else:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

from aerospace_rag.notebook_runtime import ensure_valid_cwd, git_output

PROJECT_ROOT = ensure_valid_cwd(DEFAULT_COLAB_ROOT, GITHUB_REPO_URL, in_colab=False)
REPO_BRANCH = git_output("rev-parse", "--abbrev-ref", "HEAD")
REPO_COMMIT = git_output("rev-parse", "HEAD")
print("Project root:", PROJECT_ROOT)
print("In Colab:", IN_COLAB)
print("Repo URL:", GITHUB_REPO_URL)
print("Git branch:", REPO_BRANCH)
print("Git commit:", REPO_COMMIT)

Policy: Project source is cloned fresh for each Colab run.
Running git clone: https://github.com/NAMUORI00/aerospace-rag.git
Project root: /content/aerospace-rag
In Colab: True
Repo URL: https://github.com/NAMUORI00/aerospace-rag.git
Git branch: main
Git commit: aeeff7e931bc5e9ee7ca3dfb8eff27c5090652c0


## 3. 의존성 설치와 버전 고정 확인

필요한 Python 패키지를 설치하고, 재현성 비교에 필요한 핵심 패키지 버전을 출력합니다.

In [3]:
from aerospace_rag.notebook_runtime import ensure_dependencies

PACKAGE_SNAPSHOT = ensure_dependencies(PROJECT_ROOT, IN_COLAB)

Installing: ['qdrant-client', 'docling', 'pypdf']
{
  "qdrant-client": "1.17.1",
  "sentence-transformers": "5.4.1",
  "docling": "2.92.0",
  "openpyxl": "3.1.5",
  "pypdf": "6.10.2",
  "ipywidgets": "7.7.1",
  "nbformat": "5.10.4"
}


## 4. 실행 설정 확정

이번 실행에서 사용할 RAG 설정을 한곳에 모읍니다. 아래 블록이 사용자가 직접 수정하는 기본 설정입니다. 모델명, Ollama 주소, 답변 provider, embedding backend, vector backend, 검색 개수, index reset 여부를 이 셀에서 먼저 확정하고 이후 모든 단계가 같은 값을 사용합니다.

In [4]:
# User-editable defaults
DATA_DIR = PROJECT_ROOT / "data"
INDEX_DIR = DATA_DIR / "index"

ANSWER_PROVIDER = "ollama"  # "ollama" or "extractive"
TOP_K = 5

OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "gemma4:e4b"
OLLAMA_API_KEY = ""

EXTRACTOR_LLM_BACKEND = "ollama"  # strict Ollama graph extraction; no local fallback

OLLAMA_KEEP_ALIVE = "10m"
OLLAMA_EXTRACT_TIMEOUT_SECONDS = 3600
OLLAMA_GENERATE_TIMEOUT_SECONDS = 3600
OLLAMA_EXTRACT_RETRIES = 1
OLLAMA_EXTRACT_REPAIR_RETRIES = 1
OLLAMA_EXTRACT_NUM_PREDICT = 4096
OLLAMA_ANSWER_NUM_PREDICT = 1024
OLLAMA_EXTRACT_MAX_CHARS = 1200

EMBED_BACKEND = "sentence_transformers"  # explicit debug mode: "hash"
EMBED_MODEL = "BAAI/bge-m3"
VECTOR_BACKEND = "qdrant"  # explicit debug mode: "json"
INDEX_RESET = True

SUPPORTED_ANSWER_PROVIDERS = {"ollama", "extractive"}
if ANSWER_PROVIDER not in SUPPORTED_ANSWER_PROVIDERS:
    raise ValueError(f"ANSWER_PROVIDER must be one of {sorted(SUPPORTED_ANSWER_PROVIDERS)}")

os.environ["AEROSPACE_EMBED_BACKEND"] = EMBED_BACKEND
os.environ["AEROSPACE_EMBED_MODEL"] = EMBED_MODEL
os.environ["AEROSPACE_VECTOR_BACKEND"] = VECTOR_BACKEND
os.environ["OLLAMA_BASE_URL"] = OLLAMA_BASE_URL
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL
os.environ["OLLAMA_API_KEY"] = OLLAMA_API_KEY
os.environ["EXTRACTOR_LLM_BACKEND"] = EXTRACTOR_LLM_BACKEND
os.environ["OLLAMA_KEEP_ALIVE"] = OLLAMA_KEEP_ALIVE
os.environ["OLLAMA_EXTRACT_TIMEOUT_SECONDS"] = str(OLLAMA_EXTRACT_TIMEOUT_SECONDS)
os.environ["OLLAMA_GENERATE_TIMEOUT_SECONDS"] = str(OLLAMA_GENERATE_TIMEOUT_SECONDS)
os.environ["OLLAMA_EXTRACT_RETRIES"] = str(OLLAMA_EXTRACT_RETRIES)
os.environ["OLLAMA_EXTRACT_REPAIR_RETRIES"] = str(OLLAMA_EXTRACT_REPAIR_RETRIES)
os.environ["OLLAMA_EXTRACT_NUM_PREDICT"] = str(OLLAMA_EXTRACT_NUM_PREDICT)
os.environ["OLLAMA_ANSWER_NUM_PREDICT"] = str(OLLAMA_ANSWER_NUM_PREDICT)
os.environ["OLLAMA_EXTRACT_MAX_CHARS"] = str(OLLAMA_EXTRACT_MAX_CHARS)

LLM_NEEDED = ANSWER_PROVIDER == "ollama" or EXTRACTOR_LLM_BACKEND == "ollama"

RUNTIME_CONFIG = {
    "DATA_DIR": str(DATA_DIR),
    "INDEX_DIR": str(INDEX_DIR),
    "ANSWER_PROVIDER": ANSWER_PROVIDER,
    "TOP_K": TOP_K,
    "OLLAMA_BASE_URL": OLLAMA_BASE_URL,
    "OLLAMA_MODEL": OLLAMA_MODEL,
    "OLLAMA_API_KEY_SET": bool(OLLAMA_API_KEY),
    "EXTRACTOR_LLM_BACKEND": EXTRACTOR_LLM_BACKEND,
    "OLLAMA_KEEP_ALIVE": OLLAMA_KEEP_ALIVE,
    "OLLAMA_EXTRACT_TIMEOUT_SECONDS": OLLAMA_EXTRACT_TIMEOUT_SECONDS,
    "OLLAMA_GENERATE_TIMEOUT_SECONDS": OLLAMA_GENERATE_TIMEOUT_SECONDS,
    "OLLAMA_EXTRACT_RETRIES": OLLAMA_EXTRACT_RETRIES,
    "OLLAMA_EXTRACT_REPAIR_RETRIES": OLLAMA_EXTRACT_REPAIR_RETRIES,
    "OLLAMA_EXTRACT_NUM_PREDICT": OLLAMA_EXTRACT_NUM_PREDICT,
    "OLLAMA_ANSWER_NUM_PREDICT": OLLAMA_ANSWER_NUM_PREDICT,
    "OLLAMA_EXTRACT_MAX_CHARS": OLLAMA_EXTRACT_MAX_CHARS,
    "AEROSPACE_EMBED_BACKEND": EMBED_BACKEND,
    "AEROSPACE_EMBED_MODEL": EMBED_MODEL,
    "AEROSPACE_VECTOR_BACKEND": VECTOR_BACKEND,
    "INDEX_RESET": INDEX_RESET,
    "LLM_NEEDED": LLM_NEEDED,
}
print(json.dumps(RUNTIME_CONFIG, ensure_ascii=False, indent=2))

{
  "DATA_DIR": "/content/aerospace-rag/data",
  "INDEX_DIR": "/content/aerospace-rag/data/index",
  "ANSWER_PROVIDER": "ollama",
  "TOP_K": 5,
  "OLLAMA_BASE_URL": "http://127.0.0.1:11434",
  "OLLAMA_MODEL": "gemma4:e4b",
  "OLLAMA_API_KEY_SET": false,
  "EXTRACTOR_LLM_BACKEND": "ollama",
  "OLLAMA_KEEP_ALIVE": "10m",
  "OLLAMA_EXTRACT_TIMEOUT_SECONDS": 3600,
  "OLLAMA_GENERATE_TIMEOUT_SECONDS": 3600,
  "OLLAMA_EXTRACT_RETRIES": 1,
  "OLLAMA_EXTRACT_REPAIR_RETRIES": 1,
  "OLLAMA_EXTRACT_NUM_PREDICT": 4096,
  "OLLAMA_ANSWER_NUM_PREDICT": 1024,
  "OLLAMA_EXTRACT_MAX_CHARS": 1200,
  "AEROSPACE_EMBED_BACKEND": "sentence_transformers",
  "AEROSPACE_EMBED_MODEL": "BAAI/bge-m3",
  "AEROSPACE_VECTOR_BACKEND": "qdrant",
  "INDEX_RESET": true,
  "LLM_NEEDED": true
}


## 5. Ollama 런타임과 모델 준비

답변 생성과 strict Ollama graph extraction에서 사용할 Ollama 런타임을 준비합니다. 기본 모델은 Colab T4에서 구조화 JSON extraction 안정성과 속도의 균형을 고려해 `gemma4:e4b`를 사용합니다. 기본 설정은 1시간 timeout, JSON Schema structured output, malformed JSON repair 1회, generation limit을 적용합니다. local fallback은 사용하지 않으므로 Ollama extraction이 실패하면 인덱싱도 명확히 실패합니다. 더 큰 최신 모델이 필요하면 4번 실행 설정 셀의 `OLLAMA_BASE_URL`, `OLLAMA_MODEL`, `OLLAMA_API_KEY`에서 바꿀 수 있습니다.

In [11]:
from aerospace_rag.notebook_runtime import ensure_ollama_runtime

OLLAMA_STATUS = ensure_ollama_runtime(LLM_NEEDED, in_colab=IN_COLAB, pull_model=True)

Pulling Ollama model: gemma4:e4b
Ollama ready: http://127.0.0.1:11434 model: gemma4:e4b


## 6. 데이터 파일 준비

`data/` 폴더를 재귀적으로 훑어 지원 형식의 문서를 자동으로 찾습니다. 특정 파일명을 강제하지 않습니다. Colab에서는 파일 패널의 `/content/aerospace-rag/data` 폴더에 업무 문서를 넣은 뒤 이 셀부터 다시 실행하면 됩니다. 파일이 준비되면 파일 크기와 SHA-256 해시를 출력해 같은 데이터로 재현했는지 확인합니다.

In [6]:
from aerospace_rag.ingestion import SUPPORTED_SUFFIXES
from aerospace_rag.notebook_runtime import discover_data_files

DATA_MANIFEST = discover_data_files(DATA_DIR)
print("데이터 경로:", DATA_DIR)
print("지원 형식:", ", ".join(sorted(SUPPORTED_SUFFIXES)))
if not DATA_MANIFEST:
    print("위 경로에 지원 문서 파일을 넣은 뒤 이 셀을 다시 실행하세요.")
    print("Colab 파일 패널에서 aerospace-rag/data 폴더로 파일을 드래그해 넣으면 됩니다.")
    raise FileNotFoundError(f"지원되는 data 파일이 없습니다: {DATA_DIR}")

for entry in DATA_MANIFEST:
    print(f"OK {entry['name']} bytes={entry['bytes']} sha256={entry['sha256'][:16]}...")
print("Data files:", len(DATA_MANIFEST))

데이터 경로: /content/aerospace-rag/data
지원 형식: .docx, .jpeg, .jpg, .md, .pdf, .png, .pptx, .txt, .webp, .xlsm, .xlsx
OK 251222_H3 8호기 발사 경과.pdf bytes=954268 sha256=7c60ab756d11c6dd...
OK NASA awards Momentus contract for solar sail demonstration study(영).pdf bytes=1151063 sha256=e1cabbab9f847aca...
OK 위성영상가격.png bytes=517771 sha256=db3e63e447fa8523...
OK 인공위성_질문응답.xlsx bytes=18050 sha256=389993a51bd755cd...
OK 해외정부 우주항공 현황.png bytes=219131 sha256=4bc92cbcc8f4272b...
Data files: 5


## 7. 수집/파싱 단독 확인

인덱스를 만들기 전에 입력 파일이 어떤 chunk로 변환되는지 확인합니다. 이 단계가 통과하면 데이터 파일명, 파일 형식, 기본 파서가 정상이라는 뜻입니다.

In [7]:
from collections import Counter
from aerospace_rag.ingestion import ingest_data

chunks = ingest_data(DATA_DIR, strict_expected=False)
CHUNK_SUMMARY = {
    "total_chunks": len(chunks),
    "by_file": dict(Counter(chunk.source_file for chunk in chunks)),
    "by_modality": dict(Counter(chunk.modality for chunk in chunks)),
}
print(json.dumps(CHUNK_SUMMARY, ensure_ascii=False, indent=2))

for chunk in chunks[:5]:
    loc = f"p.{chunk.page}" if chunk.page else f"{chunk.sheet}:{chunk.row}" if chunk.row else "table"
    preview = chunk.text.replace("\n", " ")[:240]
    print(f"- {chunk.chunk_id} [{chunk.source_file} / {loc} / {chunk.modality}]")
    print(" ", preview)

{
  "total_chunks": 28,
  "by_file": {
    "251222_H3 8호기 발사 경과.pdf": 2,
    "NASA awards Momentus contract for solar sail demonstration study(영).pdf": 3,
    "위성영상가격.png": 1,
    "인공위성_질문응답.xlsx": 21,
    "해외정부 우주항공 현황.png": 1
  },
  "by_modality": {
    "text": 5,
    "table": 2,
    "qa": 21
  }
}
- 251222_H3 8호기 발사 경과#t1-ac6c7ab59ac73111 [251222_H3 8호기 발사 경과.pdf / p.1 / text]
  - 1 - H3 발사 동향 보고(’25.12.26(금).,우주수송부문)□ H-3 8차 발사○(목적)일본위성항법시스템미치비키(QZS-5)위성발사○(일정및장소)‘25년12월22일(월),10시51분,다네가시마LP2○(발사체)H3-22S*소모성발사체 * (고체)부스터 SRB-3엔진 2기, (액체수소+액체산소)1단 LE-9엔진 2기, 2단 LE-5B-3엔진 1기○(결과)2단엔진재점화후조기종료되어궤도투입실패 ○(원인분석)페어링분리시이상진
- 251222_H3 8호기 발사 경과#t2-04413491f8019054 [251222_H3 8호기 발사 경과.pdf / p.2 / text]
  - 2 - 참고1 발사체 규격 및 발생 이벤트□ 발사체 규격H-3(H3-22S) 8호기 내용1단고체부스터2단위성 페어링길이 [m]약 37약 15약 12약 10.4직경 [m]약 5.2약 2.5약 5.2약 5.2중량(t)약 240약 152.4약 28약 1.8추력 [kN]약 2,942(2기)약 4,600(2기)약 137-연소시간(s)약 300약110약694-추진제LH2/LOX 복합 추진제LH2/LOX -□ 발사 계획
- NASA awards Momentus contract for solar sail demonstratio

## 8. 인덱스 생성

동일 데이터에서 Qdrant local mode, BM25, graph-lite helper index를 다시 생성합니다. `reset=True`라 이전 index 산출물은 재사용하지 않습니다.

In [8]:
from aerospace_rag.pipeline import build_index

build_started = time.perf_counter()
result = build_index(data_dir=DATA_DIR, index_dir=INDEX_DIR, reset=INDEX_RESET, strict_expected=False)
INDEX_BUILD_SECONDS = round(time.perf_counter() - build_started, 3)

INDEX_ARTIFACTS = {
    "qdrant": INDEX_DIR / "qdrant",
    "graph_index": result.graph_index_path,
    "bm25": result.bm25_path,
    "chunks": result.chunks_path,
}
ARTIFACT_REPORT = {
    name: {
        "path": str(path),
        "exists": path.exists(),
        "files": sum(1 for _ in path.rglob("*")) if path.is_dir() else 1 if path.exists() else 0,
    }
    for name, path in INDEX_ARTIFACTS.items()
}
missing_artifacts = [name for name, info in ARTIFACT_REPORT.items() if not info["exists"]]
if missing_artifacts:
    raise FileNotFoundError(f"Missing index artifacts: {missing_artifacts}")

BUILD_REPORT = {
    "data_dir": str(result.data_dir),
    "index_dir": str(result.index_dir),
    "file_count": result.file_count,
    "chunk_count": result.chunk_count,
    "qdrant_collection": result.qdrant_collection,
    "graph_index_path": str(result.graph_index_path),
    "bm25_path": str(result.bm25_path),
    "chunks_path": str(result.chunks_path),
    "artifacts": ARTIFACT_REPORT,
    "reset": INDEX_RESET,
    "seconds": INDEX_BUILD_SECONDS,
}
print(json.dumps(BUILD_REPORT, ensure_ascii=False, indent=2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

{
  "data_dir": "/content/aerospace-rag/data",
  "index_dir": "/content/aerospace-rag/data/index",
  "file_count": 5,
  "chunk_count": 28,
  "qdrant_collection": "aerospace_chunks",
  "graph_index_path": "/content/aerospace-rag/data/index/graph/graph_index.json",
  "bm25_path": "/content/aerospace-rag/data/index/bm25.json",
  "chunks_path": "/content/aerospace-rag/data/index/chunks.jsonl",
  "artifacts": {
    "qdrant": {
      "path": "/content/aerospace-rag/data/index/qdrant",
      "exists": true,
      "files": 5
    },
    "graph_index": {
      "path": "/content/aerospace-rag/data/index/graph/graph_index.json",
      "exists": true,
      "files": 1
    },
    "bm25": {
      "path": "/content/aerospace-rag/data/index/bm25.json",
      "exists": true,
      "files": 1
    },
    "chunks": {
      "path": "/content/aerospace-rag/data/index/chunks.jsonl",
      "exists": true,
      "files": 1
    }
  },
  "reset": true,
  "seconds": 772.063
}


## 8A. 도메인 데이터베이스 저장 구조 이해

인덱스 생성이 끝나면 같은 원문 chunk가 네 가지 로컬 산출물로 나뉘어 저장됩니다. 이 구조를 보면 RAG가 문서를 단순히 파일로 보관하는 것이 아니라, 답변에 필요한 검색 경로별 데이터베이스를 동시에 만든다는 점을 확인할 수 있습니다.

| 산출물 | 역할 |
| --- | --- |
| `chunks.jsonl` | 원문 조각의 표준 payload 저장소입니다. 모든 검색 채널이 다시 참조할 `chunk_id`, 본문, 원본 파일명, page/sheet/row, modality, tier 정보가 들어갑니다. |
| `qdrant` | dense text vector, dense image vector, sparse vector, chunk payload를 함께 저장하는 local Qdrant 저장소입니다. 의미 기반 검색과 이미지 modality 검색의 기준점입니다. |
| `bm25.json` | chunk별 tokenized keyword 문서를 저장합니다. 고유 명칭, 숫자, 표기 그대로의 키워드가 중요한 질문에 유리합니다. |
| `graph/graph_index.json` | entity, relations, entity_to_chunks, entity_neighbors, chunk payload를 저장합니다. 기관/위성/계약처럼 관계를 따라가야 하는 질문에 유리합니다. |

아래 셀은 산출물 존재 여부와 파일 수, chunk payload 샘플, modality 분포, BM25 token 문서, graph schema, Qdrant local collection 정보를 한국어 표와 JSON preview로 출력합니다. 또한 `graph_index.json`의 entity, relation, entity_to_chunks를 노드와 엣지 형태로 시각화해 지식 데이터베이스가 어떤 연결 구조를 갖는지 보여줍니다.

In [9]:
import html as html_lib
from collections import Counter


def _artifact_file_count(path):
    if path.is_dir():
        return sum(1 for child in path.rglob("*") if child.is_file())
    return 1 if path.exists() else 0


def _short_text(value, max_chars=220):
    text = " ".join(str(value or "").split())
    return text if len(text) <= max_chars else text[: max_chars - 1].rstrip() + "..."


def _knowledge_graph_html(graph_payload, chunk_payloads):
    entity_to_chunks = dict(graph_payload.get("entity_to_chunks") or {})
    entity_text = dict(graph_payload.get("entity_text") or {})
    entity_types = dict(graph_payload.get("entity_types") or {})
    chunk_lookup = dict(graph_payload.get("chunks") or {})
    if not chunk_lookup:
        chunk_lookup = {payload.get("chunk_id"): payload for payload in chunk_payloads if payload.get("chunk_id")}

    entity_nodes = []
    chunk_nodes = []
    entity_chunk_edges = []
    seen_chunks = set()
    for entity_id, chunk_ids in list(entity_to_chunks.items())[:6]:
        label = entity_text.get(entity_id, entity_id)
        entity_nodes.append(
            {
                "id": entity_id,
                "label": _short_text(label, 28),
                "type": entity_types.get(entity_id, "Entity"),
                "chunk_count": len(chunk_ids),
            }
        )
        for chunk_id in chunk_ids[:2]:
            payload = chunk_lookup.get(chunk_id, {})
            if chunk_id not in seen_chunks:
                seen_chunks.add(chunk_id)
                chunk_nodes.append(
                    {
                        "id": chunk_id,
                        "label": _short_text(payload.get("source_file") or chunk_id, 34),
                        "modality": payload.get("modality", "chunk"),
                    }
                )
            entity_chunk_edges.append((entity_id, chunk_id))
            if len(chunk_nodes) >= 6:
                break
        if len(chunk_nodes) >= 6:
            break

    relation_edges = []
    for relation in list(graph_payload.get("relations") or [])[:6]:
        source = str(relation.get("source") or "")
        target = str(relation.get("target") or "")
        rel_type = str(relation.get("type") or "RELATED_TO")
        if source and target:
            relation_edges.append((source, rel_type, target))

    def node_html(node, kind):
        title = html_lib.escape(str(node.get("label") or node.get("id") or "node"))
        meta = html_lib.escape(str(node.get("type") or node.get("modality") or kind))
        count = node.get("chunk_count")
        if count is not None:
            meta += f" · chunks {html_lib.escape(str(count))}"
        return (
            f"<div class='kg-node kg-{kind}'>"
            f"<div class='kg-node-title'>{title}</div>"
            f"<div class='kg-node-meta'>{meta}</div>"
            "</div>"
        )

    artifact_nodes = [
        {"label": "chunks.jsonl", "type": "표준 chunk payload"},
        {"label": "qdrant", "type": "dense/sparse vector + payload"},
        {"label": "bm25.json", "type": "tokenized keyword documents"},
        {"label": "graph_index.json", "type": "entity / relation / evidence"},
    ]
    entity_html = "".join(node_html(node, "entity") for node in entity_nodes) or "<div class='kg-empty'>entity 노드가 없습니다.</div>"
    chunk_html = "".join(node_html(node, "chunk") for node in chunk_nodes) or "<div class='kg-empty'>chunk 노드가 없습니다.</div>"
    artifact_html = "".join(node_html(node, "artifact") for node in artifact_nodes)
    entity_edge_html = "".join(
        "<div class='kg-edge'>"
        + html_lib.escape(entity_text.get(entity_id, entity_id))
        + " <span>entity_to_chunks</span> "
        + html_lib.escape(_short_text(chunk_id, 42))
        + "</div>"
        for entity_id, chunk_id in entity_chunk_edges[:8]
    ) or "<div class='kg-empty'>entity_to_chunks edge가 없습니다.</div>"
    relation_edge_html = "".join(
        "<div class='kg-edge kg-relation-edge'>"
        + html_lib.escape(entity_text.get(source, source))
        + " <span>" + html_lib.escape(rel_type) + "</span> "
        + html_lib.escape(entity_text.get(target, target))
        + "</div>"
        for source, rel_type, target in relation_edges
    ) or "<div class='kg-empty'>relation edge가 없습니다.</div>"

    return f"""
<style>
.kg-wrap {{ font-family: Arial, sans-serif; border:1px solid #d0d7de; border-radius:8px; padding:14px; margin:10px 0 18px; background:#fff; }}
.kg-title {{ font-weight:700; font-size:16px; color:#24292f; margin-bottom:4px; }}
.kg-subtitle {{ color:#57606a; font-size:13px; margin-bottom:12px; }}
.kg-layer {{ display:grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap:10px; margin:10px 0; }}
.kg-node {{ border:1px solid #d0d7de; border-radius:999px; padding:10px 12px; min-height:46px; background:#f6f8fa; box-shadow:0 1px 2px rgba(31,35,40,.06); }}
.kg-artifact {{ background:#eef6ff; border-color:#9ec5fe; }}
.kg-entity {{ background:#dafbe1; border-color:#8ddb8c; }}
.kg-chunk {{ background:#fff8c5; border-color:#eac54f; }}
.kg-node-title {{ font-weight:700; color:#24292f; font-size:13px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }}
.kg-node-meta {{ color:#57606a; font-size:12px; margin-top:3px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }}
.kg-edge-panel {{ display:grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap:10px; margin:8px 0; }}
.kg-edge-box {{ border-left:3px solid #8c959f; background:#f6f8fa; padding:8px 10px; }}
.kg-edge-box-title {{ font-weight:700; font-size:12px; color:#57606a; margin-bottom:6px; }}
.kg-edge {{ font-size:12px; color:#24292f; margin:4px 0; }}
.kg-edge span {{ color:#8250df; font-weight:700; }}
.kg-empty {{ color:#6e7781; font-size:12px; }}
</style>
<div class='kg-wrap'>
  <div class='kg-title'>지식 데이터베이스 노드 보기</div>
  <div class='kg-subtitle'>인덱스 산출물, entity 노드, chunk 노드, relation edge가 어떻게 연결되는지 `graph_index.json` 샘플로 확인합니다.</div>
  <div class='kg-layer'>{artifact_html}</div>
  <div class='kg-edge-panel'>
    <div class='kg-edge-box'><div class='kg-edge-box-title'>entity_to_chunks edge</div>{entity_edge_html}</div>
    <div class='kg-edge-box'><div class='kg-edge-box-title'>relation edge</div>{relation_edge_html}</div>
  </div>
  <div class='kg-layer'>{entity_html}</div>
  <div class='kg-layer'>{chunk_html}</div>
</div>
"""


def _html_table(rows, columns):
    header = "".join(
        "<th style='padding:8px; border-bottom:1px solid #d0d7de; text-align:left; background:#f6f8fa;'>"
        + html_lib.escape(column)
        + "</th>"
        for column in columns
    )
    body = []
    for row in rows:
        cells = "".join(
            "<td style='padding:8px; border-bottom:1px solid #eaecf0; vertical-align:top;'>"
            + html_lib.escape(str(row.get(column, "-")))
            + "</td>"
            for column in columns
        )
        body.append("<tr>" + cells + "</tr>")
    return (
        "<div style='overflow-x:auto; margin:8px 0 16px;'>"
        "<table style='border-collapse:collapse; width:100%; font-size:14px; line-height:1.5;'>"
        f"<thead><tr>{header}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )


chunks_path = INDEX_ARTIFACTS["chunks"]
bm25_path = INDEX_ARTIFACTS["bm25"]
graph_path = INDEX_ARTIFACTS["graph_index"]
qdrant_path = INDEX_ARTIFACTS["qdrant"]

storage_rows = [
    {
        "산출물": "chunks.jsonl",
        "역할": "원문 조각의 표준 payload 저장소",
        "주요 저장 내용": "chunk_id, text, source_file, modality, page/sheet/row, tier, asset refs",
        "존재": chunks_path.exists(),
        "파일 수": _artifact_file_count(chunks_path),
        "경로": str(chunks_path),
    },
    {
        "산출물": "qdrant",
        "역할": "dense text/image vector, sparse vector, payload 저장소",
        "주요 저장 내용": f"collection={BUILD_REPORT.get('qdrant_collection', 'unknown')}, point payload는 chunks.jsonl payload와 같은 기준",
        "존재": qdrant_path.exists(),
        "파일 수": _artifact_file_count(qdrant_path),
        "경로": str(qdrant_path),
    },
    {
        "산출물": "bm25.json",
        "역할": "chunk별 tokenized keyword 문서 저장소",
        "주요 저장 내용": "chunk_ids, documents[token list]",
        "존재": bm25_path.exists(),
        "파일 수": _artifact_file_count(bm25_path),
        "경로": str(bm25_path),
    },
    {
        "산출물": "graph/graph_index.json",
        "역할": "entity, relations, entity_to_chunks, neighbor index 저장소",
        "주요 저장 내용": "entity_text, entity_types, relations, entity_to_chunks, entity_neighbors, chunks",
        "존재": graph_path.exists(),
        "파일 수": _artifact_file_count(graph_path),
        "경로": str(graph_path),
    },
]
display(HTML(_html_table(storage_rows, ["산출물", "역할", "주요 저장 내용", "존재", "파일 수", "경로"])))

chunk_payloads = []
if chunks_path.exists():
    with chunks_path.open("r", encoding="utf-8") as f:
        chunk_payloads = [json.loads(line) for line in f if line.strip()]

chunk_preview = []
for payload in chunk_payloads[:2]:
    chunk_preview.append(
        {
            "chunk_id": payload.get("chunk_id"),
            "source_file": payload.get("source_file"),
            "modality": payload.get("modality"),
            "page": payload.get("page"),
            "sheet": payload.get("sheet"),
            "row": payload.get("row"),
            "tier": payload.get("tier"),
            "text_preview": _short_text(payload.get("text")),
        }
    )
modality_counts = dict(Counter(str(payload.get("modality") or "unknown") for payload in chunk_payloads))

bm25_payload = json.loads(bm25_path.read_text(encoding="utf-8")) if bm25_path.exists() else {}
bm25_documents = list(bm25_payload.get("documents") or [])
bm25_doc_lengths = [len(doc) for doc in bm25_documents]
bm25_summary = {
    "문서 수": len(bm25_documents),
    "평균 토큰 길이": round(sum(bm25_doc_lengths) / max(1, len(bm25_doc_lengths)), 2),
    "샘플 token": bm25_documents[0][:12] if bm25_documents else [],
}

graph_payload = json.loads(graph_path.read_text(encoding="utf-8")) if graph_path.exists() else {}
entity_to_chunks = dict(graph_payload.get("entity_to_chunks") or {})
entity_text = dict(graph_payload.get("entity_text") or {})
entity_to_chunks_sample = [
    {
        "entity_id": entity_id,
        "entity_label": entity_text.get(entity_id, entity_id),
        "chunks": chunk_ids[:3],
    }
    for entity_id, chunk_ids in list(entity_to_chunks.items())[:3]
]
graph_summary = {
    "schema_version": graph_payload.get("schema_version"),
    "entity 수": len(entity_to_chunks),
    "relation 수": len(graph_payload.get("relations") or []),
    "entity_to_chunks 샘플": entity_to_chunks_sample,
}

KNOWLEDGE_GRAPH_HTML = _knowledge_graph_html(graph_payload, chunk_payloads)
display(HTML(KNOWLEDGE_GRAPH_HTML))

qdrant_json_path = qdrant_path / "vectors.json"
qdrant_debug_payload_sample = []
if qdrant_json_path.exists():
    qdrant_rows = json.loads(qdrant_json_path.read_text(encoding="utf-8"))
    for row in qdrant_rows[:2]:
        payload = dict(row.get("payload") or {})
        qdrant_debug_payload_sample.append(
            {
                "id": row.get("id"),
                "payload_keys": sorted(payload.keys())[:16],
                "chunk_id": payload.get("chunk_id"),
                "source_file": payload.get("source_file"),
            }
        )
qdrant_summary = {
    "collection_name": BUILD_REPORT.get("qdrant_collection"),
    "저장 경로": str(qdrant_path),
    "payload 설명": "Qdrant point payload는 chunks.jsonl의 표준 payload를 그대로 담아 검색 결과가 원문 위치와 source로 돌아갈 수 있게 합니다.",
    "로컬 파일 수": _artifact_file_count(qdrant_path),
    "json_debug_payload_sample": qdrant_debug_payload_sample,
}

DATABASE_PREVIEW = {
    "chunks.jsonl": {
        "chunk 수": len(chunk_payloads),
        "modality별 chunk 수": modality_counts,
        "payload 샘플 2개": chunk_preview,
    },
    "bm25.json": bm25_summary,
    "graph_index.json": graph_summary,
    "qdrant": qdrant_summary,
}

display(Markdown("### 저장 구조 JSON preview\n\n```json\n" + json.dumps(DATABASE_PREVIEW, ensure_ascii=False, indent=2) + "\n```"))

산출물,역할,주요 저장 내용,존재,파일 수,경로
chunks.jsonl,원문 조각의 표준 payload 저장소,"chunk_id, text, source_file, modality, page/sheet/row, tier, asset refs",True,1,/content/aerospace-rag/data/index/chunks.jsonl
qdrant,"dense text/image vector, sparse vector, payload 저장소","collection=aerospace_chunks, point payload는 chunks.jsonl payload와 같은 기준",True,3,/content/aerospace-rag/data/index/qdrant
bm25.json,chunk별 tokenized keyword 문서 저장소,"chunk_ids, documents[token list]",True,1,/content/aerospace-rag/data/index/bm25.json
graph/graph_index.json,"entity, relations, entity_to_chunks, neighbor index 저장소","entity_text, entity_types, relations, entity_to_chunks, entity_neighbors, chunks",True,1,/content/aerospace-rag/data/index/graph/graph_index.json


### 저장 구조 JSON preview

```json
{
  "chunks.jsonl": {
    "chunk 수": 28,
    "modality별 chunk 수": {
      "text": 5,
      "table": 2,
      "qa": 21
    },
    "payload 샘플 2개": [
      {
        "chunk_id": "251222_H3 8호기 발사 경과#t1-ac6c7ab59ac73111",
        "source_file": "251222_H3 8호기 발사 경과.pdf",
        "modality": "text",
        "page": 1,
        "sheet": null,
        "row": null,
        "tier": "public",
        "text_preview": "- 1 - H3 발사 동향 보고(’25.12.26(금).,우주수송부문)□ H-3 8차 발사○(목적)일본위성항법시스템미치비키(QZS-5)위성발사○(일정및장소)‘25년12월22일(월),10시51분,다네가시마LP2○(발사체)H3-22S*소모성발사체 * (고체)부스터 SRB-3엔진 2기, (액체수소+액체산소)1단 LE-9엔진 2기, 2단 LE-5B-3엔진 1기○(결과)2단엔진재점화후조기종료되어궤도..."
      },
      {
        "chunk_id": "251222_H3 8호기 발사 경과#t2-04413491f8019054",
        "source_file": "251222_H3 8호기 발사 경과.pdf",
        "modality": "text",
        "page": 2,
        "sheet": null,
        "row": null,
        "tier": "public",
        "text_preview": "- 2 - 참고1 발사체 규격 및 발생 이벤트□ 발사체 규격H-3(H3-22S) 8호기 내용1단고체부스터2단위성 페어링길이 [m]약 37약 15약 12약 10.4직경 [m]약 5.2약 2.5약 5.2약 5.2중량(t)약 240약 152.4약 28약 1.8추력 [kN]약 2,942(2기)약 4,600(2기)약 137-연소시간(s)약 300약110약694-추진제LH2/LOX 복합 추진제LH2/..."
      }
    ]
  },
  "bm25.json": {
    "문서 수": 28,
    "평균 토큰 길이": 241.5,
    "샘플 token": [
      "1",
      "h3",
      "발사",
      "동향",
      "보고",
      "25",
      "12",
      "26",
      "금",
      "우주수송부문",
      "우주",
      "주수"
    ]
  },
  "graph_index.json": {
    "schema_version": "aerospace_graph_v2",
    "entity 수": 198,
    "relation 수": 90,
    "entity_to_chunks 샘플": [
      {
        "entity_id": "h3",
        "entity_label": "H3",
        "chunks": [
          "251222_H3 8호기 발사 경과#t1-ac6c7ab59ac73111"
        ]
      },
      {
        "entity_id": "h3-22s",
        "entity_label": "H3-22S",
        "chunks": [
          "251222_H3 8호기 발사 경과#t1-ac6c7ab59ac73111",
          "251222_H3 8호기 발사 경과#t2-04413491f8019054"
        ]
      },
      {
        "entity_id": "qzs-5",
        "entity_label": "미치비키(QZS-5)",
        "chunks": [
          "251222_H3 8호기 발사 경과#t1-ac6c7ab59ac73111"
        ]
      }
    ]
  },
  "qdrant": {
    "collection_name": "aerospace_chunks",
    "저장 경로": "/content/aerospace-rag/data/index/qdrant",
    "payload 설명": "Qdrant point payload는 chunks.jsonl의 표준 payload를 그대로 담아 검색 결과가 원문 위치와 source로 돌아갈 수 있게 합니다.",
    "로컬 파일 수": 3,
    "json_debug_payload_sample": []
  }
}
```

## 8B. Qdrant / Graph 저장 구조 시각화

이 셀은 8번 인덱스 생성 이후 만들어진 로컬 RAG 저장소가 실제로 어떤 파일과 데이터 구조로 구성되는지 확인합니다.

- Qdrant collection의 dense/sparse vector 설정과 point payload 샘플을 확인합니다.
- `chunks.jsonl`, `bm25.json`, `graph/graph_index.json`의 역할, 파일 수, 크기를 비교합니다.
- graph index의 entity, relation, entity_to_chunks 연결을 SVG 맵으로 시각화합니다.

인덱스 생성 셀을 실행한 뒤 이 셀을 실행하세요.

In [3]:
# Qdrant point + graph_index ?? ?? ???
from aerospace_rag.notebook_runtime import format_storage_visualization

for section in format_storage_visualization(INDEX_DIR, build_report=BUILD_REPORT):
    if section["type"] == "markdown":
        display(Markdown(section["content"]))
    else:
        display(HTML(section["content"]))


### Qdrant / Graph storage visualization

artifact,exists,files,size,path,role
qdrant,True,3,628.7 KB,/content/aerospace-rag/data/index/qdrant,dense/sparse vectors + chunk payload point storage
chunks.jsonl,True,1,42.3 KB,/content/aerospace-rag/data/index/chunks.jsonl,canonical chunk payload store
bm25.json,True,1,113.4 KB,/content/aerospace-rag/data/index/bm25.json,tokenized sparse keyword index
graph/graph_index.json,True,1,149.8 KB,/content/aerospace-rag/data/index/graph/graph_index.json,"entity, relation, entity_to_chunks graph store"


#### Qdrant collection metadata / vector settings

collection,collections_on_disk,points_count,vectors_count,vector_config,sparse_config
aerospace_chunks,aerospace_chunks,28,,"{'dense_text': VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'dense_image': VectorParams(size=1024, dist...","{'sparse': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}"


#### Qdrant point payload sample

...

#### Graph index summary

schema_version,entities,relations,chunks,entity_to_chunk_edges
aerospace_graph_v2,198,90,28,208


#### Top graph entities

entity_id,label,type,chunk_count,sample_chunks
nasa,NASA,Organization,5,"NASA awards Momentus contract for solar sail..., NASA awards Momentus contract for solar sail..., NASA awards Momentus contract for solar sail..."
h3-22s,H3-22S,Launch Vehicle,2,"251222_H3 8호기 발사 경과.pdf, 251222_H3 8호기 발사 경과.pdf"
k2,K2,위성/모드,2,"위성영상가격.png, 인공위성_질문응답.xlsx"
k3,K3,위성/모드,2,"위성영상가격.png, 인공위성_질문응답.xlsx"
k3a,K3A,위성/모드,2,"위성영상가격.png, 인공위성_질문응답.xlsx"
momentus,Momentus,Program/Contract,2,"NASA awards Momentus contract for solar sail..., NASA awards Momentus contract for solar sail..."
항우연,항우연,Organization,2,"인공위성_질문응답.xlsx, 인공위성_질문응답.xlsx"
2021,2021,YEAR,1,인공위성_질문응답.xlsx
2022,2022,YEAR,1,인공위성_질문응답.xlsx
2023,2023,YEAR,1,인공위성_질문응답.xlsx


#### Storage relationship map

## 9. 검색 단독 검증

LLM을 호출하기 전에 retrieval만 실행합니다. 답변 품질 문제가 생겼을 때 검색 문제인지 생성 문제인지 분리해서 볼 수 있습니다.

검색 채널은 질문 성격에 따라 서로 다른 강점을 가집니다.
- dense(Qdrant dense text/image): 표현이 달라도 의미가 가까운 문장, 표 설명, 이미지 설명을 찾는 데 유리합니다.
- sparse(BM25/Qdrant sparse): 위성명, 기관명, 날짜, 금액, 약어처럼 표면 키워드가 중요한 질문에 유리합니다.
- graph(graph-lite): 엔티티와 관계를 따라가야 하는 계약, 기관, 위성, 원인-결과 질문에 유리합니다.

진단 정보에서는 `channel_weights`가 채널별 결합 가중치, `query_segment`가 질문 유형 분류, `top_doc_channel_contributions`가 상위 chunk 점수에 각 채널이 기여한 값을 뜻합니다.

In [12]:
from aerospace_rag.config import Settings
from aerospace_rag.notebook_runtime import format_retrieval_markdown
from aerospace_rag.stores.local_index import LocalIndex

RETRIEVAL_QUESTION = "위성영상 가격은 저장영상과 신규촬영에서 어떻게 다른가?"
index = LocalIndex(INDEX_DIR, settings=Settings.from_env())
hits = index.search(RETRIEVAL_QUESTION, top_k=TOP_K)
display(Markdown(format_retrieval_markdown(RETRIEVAL_QUESTION, hits, index.last_diagnostics)))

### 검색 요약

- 질문: 위성영상 가격은 저장영상과 신규촬영에서 어떻게 다른가?

- 검색 결과 수: 5

- 활성 채널: bm25, graph, qdrant, vector_image

### 상위 결과
1. `위성영상가격.png` (table) score=0.016
   - channels: vector_dense_text=0.009, vector_sparse=0.007
   - excerpt: | 구분 | 위성/모드 | 해상도(m) | 저장영상(AO) | 신규촬영(NTO) | | --- | --- | ---: | --- | --- | | EO | K2 | 1 | 15 x 15 scene, $900, 1,260,000원 | 신규촬영 없음 | | EO | K3 | 0.7 | 16 x 16 scene, $2,048, 2,867,200원, km^2 $8 | 16 x 16 scene, $4,096, 5,734,400원, km^2 $16 | | EO | K3A | 0.55 | 13 x 13 sc…
2. `인공위성_질문응답.xlsx` (Sheet1:3) score=0.016
   - channels: vector_dense_text=0.009, vector_sparse=0.007
   - excerpt: 질문: 위성영상 표준가격 결정 답변: 해외 유사 스펙의 타 위성영상 가격을 고려한 위성영상 판매사, Reseller에 대한 시장조사 수행 결과를 바탕으로 사업발주처(현재 항우연)와 판매대행업체가 표준가격(10% 할인) 결정 후 촉진위 승인. 현재, 판매대행사는 정해진 표준가격의 추가로 최대 90% 할인 가격으로 판매 중* * 판매대행사별, 위성별, 상황에 따라 다름
3. `인공위성_질문응답.xlsx` (Sheet1:22) score=0.016
   - channels: vector_dense_text=0.009, vector_sparse=0.007
   - excerpt: 질문: 사용자그룹과 협의체 차이 답변: 사용자그룹 * 항우연에서 무상/실비로 제공하는 기관을 등록(대학중심, 연구기관은 실비(일부 무상제공)) * 현재는 다목적시리즈만 제공, 남북 접경지역(서해5도 포함) 및 북한지역 제공 제한 * 북한지역은 수요처 사전 검토 후 제한적으로 제공 * 정밀정사영상 등과 같은 공개제한 영상 미제공 협의체 기관 * 우주청에서 협의체 가입을 승인한 기관으로 무상 제공 * 현재 다목적, 차중1호(표준영상), 페루셋 제공 중 * 남북 접경지역(공개제한) 및 북한지역 모두…
4. `인공위성_질문응답.xlsx` (Sheet1:5) score=0.016
   - channels: vector_dense_text=0.009, vector_sparse=0.007
   - excerpt: 질문: 25년 기준, 판매대행사와 추가 계약건이 있는지? 답변: 고부가영상제품* 판매의 경우, 생산능력을 확보하는 것은 기업체가 수행하지만, 계약조건은 사업발주처와 협의 필요 * 표준영상을 원격탐사 소프트웨어 또는 활용솔루션을 이용해 가공한 결과물 차세대 중용위성 시리즈 상용화 개시 시점에 본 사업을 통해 선정된 판매대행사와 우선 협상을 추진할 계획임
5. `인공위성_질문응답.xlsx` (Sheet1:16) score=0.015
   - channels: vector_dense_text=0.008, vector_sparse=0.007
   - excerpt: 질문: 다목적실용위성 5호의 영상처리 답변: L0 Raw L1A SCS Single Look Complex L1B DSM Multi-look Data L1C GEC Georeferenced Data L1D GTC GEC+DEM

<details>
<summary>Retrieval diagnostics</summary>

```json
{
  "channels": [
    "bm25",
    "graph",
    "qdrant",
    "vector_image"
  ],
  "fusion": {
    "channel_weights": {
      "vector_dense_text": 0.55,
      "vector_sparse": 0.45,
      "vector_image": 0.0,
      "graph": 0.0,
      "qdrant": 0.55,
      "bm25": 0.45
    },
    "channel_enabled": {
      "vector_dense_text": true,
      "vector_sparse": true,
      "vector_image": true,
      "graph": true
    },
    "top_doc_channel_contributions": {
      "위성영상가격#t0-52f5c67820b371b9": {
        "vector_dense_text": 0.009016393442622951,
        "vector_sparse": 0.007377049180327869
      },
      "인공위성_질문응답#q3-198a7662452ee8ac": {
        "vector_dense_text": 0.008870967741935484,
        "vector_sparse": 0.007258064516129033
      },
      "인공위성_질문응답#q22-92dec863650071ea": {
        "vector_dense_text": 0.00859375,
        "vector_sparse": 0.007142857142857143
      },
      "인공위성_질문응답#q5-afb6e9e64fad55d3": {
        "vector_dense_text": 0.00873015873015873,
        "vector_sparse": 0.006923076923076923
      },
      "인공위성_질문응답#q16-d8e3a63590fc47c0": {
        "vector_dense_text": 0.008088235294117648,
        "vector_sparse": 0.00703125
      },
      "인공위성_질문응답#q6-d1333ace606d41ab": {
        "vector_dense_text": 0.008333333333333335,
        "vector_sparse": 0.0064285714285714285
      },
      "인공위성_질문응답#q7-5b1afc7d6c7f86f6": {
        "vector_dense_text": 0.008461538461538463,
        "vector_sparse": 0.0062499999999999995
      },
      "인공위성_질문응답#q12-f16e91ffec36c2b1": {
        "vector_dense_text": 0.0076388888888888895,
        "vector_sparse": 0.006818181818181819
      },
      "인공위성_질문응답#q20-8f51202186c486ba": {
        "vector_dense_text": 0.007746478873239438,
        "vector_sparse": 0.006617647058823529
      },
      "인공위성_질문응답#q15-82edc59ae7667384": {
        "vector_dense_text": 0.007971014492753625,
        "vector_sparse": 0.006
      }
    },
    "fusion_policy": "runtime_profile_weighted_rrf",
    "query_segment": "entity_rich",
    "weights_source": "runtime_profile",
    "weights_path": "/content/aerospace-rag/data/index/fusion_weights.runtime.json",
    "fusion_profile_scope": "default",
    "fusion_profile_id": "self-calibrated-e00ccb16839e9a93",
    "candidate_depth": 12,
    "evidence_adjustments": [],
    "selection_run_type": "main"
  },
  "embedding_provider": "sentence_transformers",
  "embedding_model": "BAAI/bge-m3"
}
```
</details>

## 10. LLM 답변 생성

동일 질문을 `ask()` 파이프라인으로 실행합니다. Ollama 호출이 실패하면 명확한 오류가 나며, LLM 없이 확인하려면 4번 설정 셀에서 `ANSWER_PROVIDER = "extractive"`로 바꿉니다. diagnostics에서 provider, retrieval 채널, weighted RRF 결합 상태를 함께 확인합니다.

In [13]:
from aerospace_rag.notebook_runtime import format_answer_markdown
from aerospace_rag.pipeline import ask

question = RETRIEVAL_QUESTION
response = ask(question, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
display(Markdown(format_answer_markdown(response)))

### 답변

표 데이터에 따르면, 위성영상 가격은 다음과 같이 다릅니다:

*   **EO/K2:** 저장영상(AO)은 $900, 1,260,000원이며, 신규촬영(NTO)은 '신규촬영 없음'입니다.
*   **EO/K3:** 저장영상(AO)은 $2,048, 2,867,200원, km^2 $8이며, 신규촬영(NTO)은 $4,096, 5,734,400원, km^2 $16입니다.
*   **EO/K3A:** 저장영상(AO)은 $1,352, 1,892,800원, km^2 $8이며, 신규촬영(NTO)은 $3,042, 4,258,800원, km^2 $18입니다.
*   **SAR/HR(UH):** 저장영상(AO)은 $900, 1,260,000원이며, 신규촬영(NTO)은 $2,800, 3,920,000원입니다.
*   **SAR/ST(ES):** 저장영상(AO)은 $600, 840,000원이며, 신규촬영(NTO)은 $1,800, 2,520,000원입니다.
*   **SAR/WS:** 저장영상(AO)은 $400, 560,000원이며, 신규촬영(NTO)은 $1,400, 1,960,000원입니다.

### 상위 근거
1. `위성영상가격.png` (table) score=0.016
   - channels: vector_dense_text=0.009, vector_sparse=0.007
2. `인공위성_질문응답.xlsx` (Sheet1:3) score=0.016
   - channels: vector_dense_text=0.009, vector_sparse=0.007
3. `인공위성_질문응답.xlsx` (Sheet1:22) score=0.016
   - channels: vector_dense_text=0.009, vector_sparse=0.007

<details>
<summary>Routing</summary>

```json
{
  "provider": "ollama",
  "retrieval": "qdrant+bm25+graph-lite"
}
```
</details>

<details>
<summary>Diagnostics</summary>

```json
{
  "channels": [
    "bm25",
    "graph",
    "qdrant",
    "vector_image"
  ],
  "source_count": 5,
  "provider": "ollama",
  "fusion": {
    "channel_weights": {
      "vector_dense_text": 0.55,
      "vector_sparse": 0.45,
      "vector_image": 0.0,
      "graph": 0.0,
      "qdrant": 0.55,
      "bm25": 0.45
    },
    "channel_enabled": {
      "vector_dense_text": true,
      "vector_sparse": true,
      "vector_image": true,
      "graph": true
    },
    "top_doc_channel_contributions": {
      "위성영상가격#t0-52f5c67820b371b9": {
        "vector_dense_text": 0.009016393442622951,
        "vector_sparse": 0.007377049180327869
      },
      "인공위성_질문응답#q3-198a7662452ee8ac": {
        "vector_dense_text": 0.008870967741935484,
        "vector_sparse": 0.007258064516129033
      },
      "인공위성_질문응답#q22-92dec863650071ea": {
        "vector_dense_text": 0.00859375,
        "vector_sparse": 0.007142857142857143
      },
      "인공위성_질문응답#q5-afb6e9e64fad55d3": {
        "vector_dense_text": 0.00873015873015873,
        "vector_sparse": 0.006923076923076923
      },
      "인공위성_질문응답#q16-d8e3a63590fc47c0": {
        "vector_dense_text": 0.008088235294117648,
        "vector_sparse": 0.00703125
      },
      "인공위성_질문응답#q6-d1333ace606d41ab": {
        "vector_dense_text": 0.008333333333333335,
        "vector_sparse": 0.0064285714285714285
      },
      "인공위성_질문응답#q7-5b1afc7d6c7f86f6": {
        "vector_dense_text": 0.008461538461538463,
        "vector_sparse": 0.0062499999999999995
      },
      "인공위성_질문응답#q12-f16e91ffec36c2b1": {
        "vector_dense_text": 0.0076388888888888895,
        "vector_sparse": 0.006818181818181819
      },
      "인공위성_질문응답#q20-8f51202186c486ba": {
        "vector_dense_text": 0.007746478873239438,
        "vector_sparse": 0.006617647058823529
      },
      "인공위성_질문응답#q15-82edc59ae7667384": {
        "vector_dense_text": 0.007971014492753625,
        "vector_sparse": 0.006
      }
    },
    "fusion_policy": "runtime_profile_weighted_rrf",
    "query_segment": "entity_rich",
    "weights_source": "runtime_profile",
    "weights_path": "/content/aerospace-rag/data/index/fusion_weights.runtime.json",
    "fusion_profile_scope": "default",
    "fusion_profile_id": "self-calibrated-e00ccb16839e9a93",
    "candidate_depth": 12,
    "evidence_adjustments": [],
    "selection_run_type": "main"
  },
  "embedding_provider": "sentence_transformers",
  "embedding_model": "BAAI/bge-m3"
}
```
</details>

## 11. 근거 확인

최종 답변에 사용된 source reference를 확인합니다. page/sheet/row와 channel score가 함께 출력됩니다.

In [14]:
from aerospace_rag.notebook_runtime import format_sources_markdown

display(Markdown(format_sources_markdown(response.sources)))

### 근거 상세

#### 1. `위성영상가격.png` (table)

- score: 0.016

- channels: vector_dense_text=0.009, vector_sparse=0.007

> | 구분 | 위성/모드 | 해상도(m) | 저장영상(AO) | 신규촬영(NTO) | | --- | --- | ---: | --- | --- | | EO | K2 | 1 | 15 x 15 scene, $900, 1,260,000원 | 신규촬영 없음 | | EO | K3 | 0.7 | 16 x 16 scene, $2,048, 2,867,200원, km^2 $8 | 16 x 16 scene, $4,096, 5,734,400원, km^2 $16 | | EO | K3A | 0.55 | 13 x 13 scene, $1,352, 1,892,800원, km^2 $8 | 13 x 13 scene, $3,042, 4,258,800원, km^2 $18...

#### 2. `인공위성_질문응답.xlsx` (Sheet1:3)

- score: 0.016

- channels: vector_dense_text=0.009, vector_sparse=0.007

> 질문: 위성영상 표준가격 결정 답변: 해외 유사 스펙의 타 위성영상 가격을 고려한 위성영상 판매사, Reseller에 대한 시장조사 수행 결과를 바탕으로 사업발주처(현재 항우연)와 판매대행업체가 표준가격(10% 할인) 결정 후 촉진위 승인. 현재, 판매대행사는 정해진 표준가격의 추가로 최대 90% 할인 가격으로 판매 중* * 판매대행사별, 위성별, 상황에 따라 다름

#### 3. `인공위성_질문응답.xlsx` (Sheet1:22)

- score: 0.016

- channels: vector_dense_text=0.009, vector_sparse=0.007

> 질문: 사용자그룹과 협의체 차이 답변: 사용자그룹 * 항우연에서 무상/실비로 제공하는 기관을 등록(대학중심, 연구기관은 실비(일부 무상제공)) * 현재는 다목적시리즈만 제공, 남북 접경지역(서해5도 포함) 및 북한지역 제공 제한 * 북한지역은 수요처 사전 검토 후 제한적으로 제공 * 정밀정사영상 등과 같은 공개제한 영상 미제공 협의체 기관 * 우주청에서 협의체 가입을 승인한 기관으로 무상 제공 * 현재 다목적, 차중1호(표준영상), 페루셋 제공 중 * 남북 접경지역(공개제한) 및 북한지역 모두 제공 * 정밀정사영상, 한반도 모자이크영상 등과 같은 공개제한 영상 제공

#### 4. `인공위성_질문응답.xlsx` (Sheet1:5)

- score: 0.016

- channels: vector_dense_text=0.009, vector_sparse=0.007

> 질문: 25년 기준, 판매대행사와 추가 계약건이 있는지? 답변: 고부가영상제품* 판매의 경우, 생산능력을 확보하는 것은 기업체가 수행하지만, 계약조건은 사업발주처와 협의 필요 * 표준영상을 원격탐사 소프트웨어 또는 활용솔루션을 이용해 가공한 결과물 차세대 중용위성 시리즈 상용화 개시 시점에 본 사업을 통해 선정된 판매대행사와 우선 협상을 추진할 계획임

#### 5. `인공위성_질문응답.xlsx` (Sheet1:16)

- score: 0.015

- channels: vector_dense_text=0.008, vector_sparse=0.007

> 질문: 다목적실용위성 5호의 영상처리 답변: L0 Raw L1A SCS Single Look Complex L1B DSM Multi-look Data L1C GEC Georeferenced Data L1D GTC GEC+DEM

## 12. 반복 질문 예시

여러 질문을 같은 index에 대해 실행해 retrieval/answer 흐름이 반복 가능하게 동작하는지 확인합니다.

In [15]:
from aerospace_rag.notebook_runtime import build_response_row, format_results_table

sample_questions = [
    "H3 8호기 발사 실패 원인은?",
    "NASA solar sail 연구의 목적은?",
    "국가별 우주항공 예산과 인력 현황은?",
    "인공위성 판매대행사 선정 절차는 얼마나 걸리는가?",
]

SAMPLE_RESULTS = []
for q in sample_questions:
    r = ask(q, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
    SAMPLE_RESULTS.append(build_response_row(q, r))

display(HTML(format_results_table(SAMPLE_RESULTS, columns=["question", "summary", "top_source", "source_count", "provider"])))

Question,Answer Summary,Top Source,Sources,Provider
H3 8호기 발사 실패 원인은?,2단 엔진 재점화 후 조기 종료되어 궤도 투입에 실패했기 때문입니다. (페어링 분리 시 이상 진동 발생으로 2단 연료 탱크 가압이 제대로 되지 않은 사유 혹은 2단 연료 탱크 누설 발생으로 추정),251222_H3 8호기 발사 경과.pdf,5,ollama
NASA solar sail 연구의 목적은?,NASA와 NOAA가 공동 개발한 태양 돛 기술은 우주 날씨 모니터링 위성이 태양에 더 가까이 운용되어 지자기 폭풍과 같은 태양 현상에 대한 조기 경보를 제공하는 것이 목적입니다.,NASA awards Momentus contract for solar sail demonstration study(영).pdf,5,ollama
국가별 우주항공 예산과 인력 현황은?,국가별 우주항공 예산과 인력 현황은 다음과 같습니다.,해외정부 우주항공 현황.png,5,ollama
인공위성 판매대행사 선정 절차는 얼마나 걸리는가?,보통 4개월 이상 소요됩니다.,인공위성_질문응답.xlsx,5,ollama


## 13. 실제 업무파일 RAG 검증

실제 `data/` 업무파일 전체로 만든 index에 대해 대표 질문을 실행합니다. 각 질문마다 provider, 검색 채널, top source, 고유 source 목록, 답변 preview를 출력해 Colab에서도 ingest부터 답변 생성까지 한 번에 검증할 수 있습니다.

In [14]:
ACTUAL_RAG_QUESTIONS = [
    "H3 8호기 발사 경과의 핵심 내용은 무엇인가?",
    "NASA가 Momentus에 수여한 계약은 무엇인가?",
    "위성영상 가격에서 저장영상과 신규촬영은 어떻게 다른가?",
    "해외정부 우주항공 현황 문서에서 확인되는 주요 국가나 기관은 무엇인가?",
    "인공위성 질문응답 문서 기준으로 인공위성 관련 핵심 설명은 무엇인가?",
]

ACTUAL_RAG_RESULTS = []
for case_idx, q in enumerate(ACTUAL_RAG_QUESTIONS, start=1):
    r = ask(q, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
    ACTUAL_RAG_RESULTS.append(build_response_row(q, r, case=case_idx))

display(HTML(format_results_table(ACTUAL_RAG_RESULTS, columns=["case", "question", "summary", "top_source", "source_count", "channels", "source_files"])))

Case,Question,Answer Summary,Top Source,Sources,Channels,Source Files
1,H3 8호기 발사 경과의 핵심 내용은 무엇인가?,H3 8호기 발사 경과의 핵심 내용은 다음과 같습니다.,251222_H3 8호기 발사 경과.pdf,5,"bm25, graph, qdrant, vector_image","251222_H3 8호기 발사 경과.pdf, 인공위성_질문응답.xlsx, 해외정부 우주항공 현황.png"
2,NASA가 Momentus에 수여한 계약은 무엇인가?,NASA가 Momentus에 수여한 계약은 태양돛 시연 연구(solar sail demonstration study)에 관한 것입니다.,NASA awards Momentus contract for solar sail demonstration study(영).pdf,5,"bm25, graph, qdrant, vector_image","NASA awards Momentus contract for solar sail demonstration study(영).pdf, 인공위성_질문응답.xlsx"
3,위성영상 가격에서 저장영상과 신규촬영은 어떻게 다른가?,"표 데이터에 따르면, 위성영상 가격에서 저장영상(AO)과 신규촬영(NTO)은 다음과 같이 다릅니다: EO | K3:",위성영상가격.png,5,"bm25, graph, qdrant, vector_image","위성영상가격.png, 인공위성_질문응답.xlsx"
4,해외정부 우주항공 현황 문서에서 확인되는 주요 국가나 기관은 무엇인가?,"표 데이터에 따르면 주요 국가나 기관은 다음과 같습니다: 국가: 미국, 러시아, 유럽, 중국, 일본, 인도, 한국",해외정부 우주항공 현황.png,5,"bm25, graph, qdrant, vector_image","해외정부 우주항공 현황.png, 인공위성_질문응답.xlsx"
5,인공위성 질문응답 문서 기준으로 인공위성 관련 핵심 설명은 무엇인가?,"인공위성 관련 핵심 설명은 다음과 같습니다: 차세대중형위성: 표준플랫폼(500kg급)을 기반으로 민간주도 위성 개발 및 활용이 가능하며, 국토, 농업, 산림, 수자원 등 분야별 공공서비스 기반으로 구축됩니다.",인공위성_질문응답.xlsx,5,"bm25, graph, qdrant, vector_image",인공위성_질문응답.xlsx
